In [1]:
"""
============================================================
  RECOMMENDATION SYSTEM
  Techniques: Collaborative Filtering + Content-Based Filtering
  Domain: Movies
============================================================
"""

import numpy as np
from collections import defaultdict
import math


# ─────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────

# User-movie ratings (1–5 scale, 0 = not rated)
ratings = {
    "Alice":   {"Inception": 5, "Interstellar": 4, "The Matrix": 5, "Avengers": 2, "Titanic": 1, "The Notebook": 1},
    "Bob":     {"Inception": 4, "Interstellar": 5, "The Matrix": 4, "Avengers": 3, "Titanic": 2, "The Notebook": 2},
    "Charlie": {"Inception": 2, "Interstellar": 1, "The Matrix": 2, "Avengers": 5, "Titanic": 4, "The Notebook": 5},
    "Diana":   {"Inception": 1, "Interstellar": 2, "The Matrix": 1, "Avengers": 5, "Titanic": 5, "The Notebook": 4},
    "Eve":     {"Inception": 5, "Interstellar": 3, "The Matrix": 4, "Avengers": 4, "Titanic": 3, "The Notebook": 2},
    "Frank":   {"Inception": 3, "Interstellar": 4, "The Matrix": 5, "Avengers": 1, "Titanic": 2, "The Notebook": 1},
}

# Movie metadata for content-based filtering
movie_features = {
    "Inception":      {"genre": ["sci-fi", "thriller"],   "director": "Nolan", "year": 2010},
    "Interstellar":   {"genre": ["sci-fi", "drama"],      "director": "Nolan", "year": 2014},
    "The Matrix":     {"genre": ["sci-fi", "action"],     "director": "Wachowski", "year": 1999},
    "Avengers":       {"genre": ["action", "superhero"],  "director": "Russo", "year": 2019},
    "Titanic":        {"genre": ["romance", "drama"],     "director": "Cameron", "year": 1997},
    "The Notebook":   {"genre": ["romance", "drama"],     "director": "Cassavetes", "year": 2004},
    "Tenet":          {"genre": ["sci-fi", "thriller"],   "director": "Nolan", "year": 2020},
    "Iron Man":       {"genre": ["action", "superhero"],  "director": "Favreau", "year": 2008},
    "La La Land":     {"genre": ["romance", "musical"],   "director": "Chazelle", "year": 2016},
    "Gravity":        {"genre": ["sci-fi", "thriller"],   "director": "Cuaron", "year": 2013},
}

ALL_MOVIES = list(movie_features.keys())


# ─────────────────────────────────────────────
#  HELPER: COSINE SIMILARITY
# ─────────────────────────────────────────────

def cosine_similarity(vec_a: dict, vec_b: dict) -> float:
    """Compute cosine similarity between two sparse rating vectors."""
    common = set(vec_a) & set(vec_b)
    if not common:
        return 0.0

    dot   = sum(vec_a[m] * vec_b[m] for m in common)
    mag_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    mag_b = math.sqrt(sum(v**2 for v in vec_b.values()))

    if mag_a == 0 or mag_b == 0:
        return 0.0
    return dot / (mag_a * mag_b)


# ─────────────────────────────────────────────
#  1. COLLABORATIVE FILTERING (User-Based)
# ─────────────────────────────────────────────

def get_similar_users(target_user: str, all_ratings: dict, top_n: int = 3) -> list:
    """Find users most similar to the target user based on ratings."""
    similarities = []
    target_ratings = all_ratings[target_user]

    for user, user_ratings in all_ratings.items():
        if user == target_user:
            continue
        sim = cosine_similarity(target_ratings, user_ratings)
        similarities.append((user, round(sim, 4)))

    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_n]


def collaborative_filtering(target_user: str, all_ratings: dict, top_n_recs: int = 3) -> list:
    """
    User-Based Collaborative Filtering.
    Predicts ratings for unseen movies using weighted average from similar users.
    """
    target_ratings = all_ratings[target_user]
    unrated_movies = [m for m in ALL_MOVIES if m not in target_ratings]

    similar_users = get_similar_users(target_user, all_ratings, top_n=len(all_ratings) - 1)

    predicted_scores = {}
    for movie in unrated_movies:
        weighted_sum  = 0.0
        similarity_sum = 0.0

        for (user, sim) in similar_users:
            if movie in all_ratings[user] and sim > 0:
                weighted_sum  += sim * all_ratings[user][movie]
                similarity_sum += sim

        if similarity_sum > 0:
            predicted_scores[movie] = round(weighted_sum / similarity_sum, 2)

    # Sort by predicted score
    sorted_recs = sorted(predicted_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_recs[:top_n_recs]


# ─────────────────────────────────────────────
#  2. CONTENT-BASED FILTERING
# ─────────────────────────────────────────────

def build_movie_vector(movie: str) -> dict:
    """Create a binary feature vector for a movie."""
    features = movie_features[movie]
    vector = {}

    for genre in features["genre"]:
        vector[f"genre_{genre}"] = 1

    vector[f"director_{features['director']}"] = 1

    # Decade as a feature
    decade = (features["year"] // 10) * 10
    vector[f"decade_{decade}"] = 1

    return vector


def movie_content_similarity(movie_a: str, movie_b: str) -> float:
    """Compute content similarity between two movies."""
    vec_a = build_movie_vector(movie_a)
    vec_b = build_movie_vector(movie_b)
    return cosine_similarity(vec_a, vec_b)


def content_based_filtering(target_user: str, all_ratings: dict, top_n_recs: int = 3) -> list:
    """
    Content-Based Filtering.
    Recommends movies similar in features to what the user already liked (rating >= 4).
    """
    user_ratings = all_ratings[target_user]
    liked_movies  = [m for m, r in user_ratings.items() if r >= 4]
    unrated_movies = [m for m in ALL_MOVIES if m not in user_ratings]

    scores = defaultdict(float)
    counts = defaultdict(int)

    for liked in liked_movies:
        for candidate in unrated_movies:
            sim = movie_content_similarity(liked, candidate)
            scores[candidate] += sim
            counts[candidate] += 1

    # Average similarity score
    avg_scores = {m: round(scores[m] / counts[m], 4) for m in scores if counts[m] > 0}
    sorted_recs = sorted(avg_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_recs[:top_n_recs]


# ─────────────────────────────────────────────
#  3. HYBRID RECOMMENDER
# ─────────────────────────────────────────────

def hybrid_recommendations(target_user: str, all_ratings: dict,
                            cf_weight: float = 0.5, cb_weight: float = 0.5,
                            top_n_recs: int = 3) -> list:
    """
    Hybrid Recommender: combines CF and CB scores.
    cf_weight + cb_weight should sum to 1.0
    """
    cf_recs = dict(collaborative_filtering(target_user, all_ratings, top_n_recs=20))
    cb_recs = dict(content_based_filtering(target_user, all_ratings, top_n_recs=20))

    all_candidates = set(cf_recs) | set(cb_recs)

    # Normalize each score set to [0, 1]
    def normalize(score_dict):
        if not score_dict:
            return {}
        max_v = max(score_dict.values())
        min_v = min(score_dict.values())
        rng   = max_v - min_v or 1
        return {k: (v - min_v) / rng for k, v in score_dict.items()}

    cf_norm = normalize(cf_recs)
    cb_norm = normalize(cb_recs)

    hybrid_scores = {}
    for movie in all_candidates:
        cf_score = cf_norm.get(movie, 0)
        cb_score = cb_norm.get(movie, 0)
        hybrid_scores[movie] = round(cf_weight * cf_score + cb_weight * cb_score, 4)

    sorted_recs = sorted(hybrid_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_recs[:top_n_recs]


# ─────────────────────────────────────────────
#  DISPLAY HELPERS
# ─────────────────────────────────────────────

def print_section(title: str):
    print("\n" + "=" * 55)
    print(f"  {title}")
    print("=" * 55)


def print_recommendations(recs: list, label: str):
    print(f"\n  📽  {label}:")
    if not recs:
        print("     No recommendations found.")
        return
    for i, (movie, score) in enumerate(recs, 1):
        bar = "█" * int(score * 10) if score <= 5 else "█" * 10
        print(f"     {i}. {movie:<20} Score: {score:.2f}  {bar}")


def print_user_profile(user: str):
    print(f"\n  👤 User: {user}")
    print(f"  {'Movie':<22} {'Rating':>6}")
    print(f"  {'-'*30}")
    for movie, rating in ratings[user].items():
        stars = "★" * rating + "☆" * (5 - rating)
        print(f"  {movie:<22} {stars}")


# ─────────────────────────────────────────────
#  MAIN DEMO
# ─────────────────────────────────────────────

def main():
    print_section("MOVIE RECOMMENDATION SYSTEM")
    print("  Techniques: User-Based CF | Content-Based | Hybrid")

    demo_users = ["Alice", "Charlie"]

    for user in demo_users:
        print_section(f"RECOMMENDATIONS FOR: {user.upper()}")
        print_user_profile(user)

        # Similar users
        similar = get_similar_users(user, ratings)
        print(f"\n  🤝 Most Similar Users:")
        for su, sim in similar:
            print(f"     • {su:<10}  similarity: {sim:.4f}")

        # Collaborative Filtering
        cf = collaborative_filtering(user, ratings)
        print_recommendations(cf, "Collaborative Filtering (User-Based)")

        # Content-Based
        cb = content_based_filtering(user, ratings)
        print_recommendations(cb, "Content-Based Filtering")

        # Hybrid
        hybrid = hybrid_recommendations(user, ratings)
        print_recommendations(hybrid, "Hybrid Recommendations (CF 50% + CB 50%)")

    # ── Extra: recommend for a new user with minimal ratings
    print_section("COLD-START: NEW USER (only 2 ratings)")
    new_user_ratings = {"newUser": {"Inception": 5, "Avengers": 4}}
    combined_ratings = {**ratings, **new_user_ratings}

    print("\n  👤 User: newUser")
    print("     Rated: Inception (5★), Avengers (4★)")

    cb_new = content_based_filtering("newUser", combined_ratings)
    print_recommendations(cb_new, "Content-Based (best for cold-start)")

    print("\n" + "=" * 55)
    print("  ✅ Done! All recommendations generated.")
    print("=" * 55 + "\n")


if __name__ == "__main__":
    main()


  MOVIE RECOMMENDATION SYSTEM
  Techniques: User-Based CF | Content-Based | Hybrid

  RECOMMENDATIONS FOR: ALICE

  👤 User: Alice
  Movie                  Rating
  ------------------------------
  Inception              ★★★★★
  Interstellar           ★★★★☆
  The Matrix             ★★★★★
  Avengers               ★★☆☆☆
  Titanic                ★☆☆☆☆
  The Notebook           ★☆☆☆☆

  🤝 Most Similar Users:
     • Frank       similarity: 0.9607
     • Bob         similarity: 0.9590
     • Eve         similarity: 0.9282

  📽  Collaborative Filtering (User-Based):
     No recommendations found.

  📽  Content-Based Filtering:
     1. Tenet                Score: 0.50  █████
     2. Gravity              Score: 0.50  █████
     3. La La Land           Score: 0.17  █

  📽  Hybrid Recommendations (CF 50% + CB 50%):
     1. Gravity              Score: 0.50  █████
     2. Tenet                Score: 0.50  █████
     3. La La Land           Score: 0.10  █

  RECOMMENDATIONS FOR: CHARLIE

  👤 User: Ch